# FLAG 2027 — NB-11: the organiser's own encoders, other layers + test-time augmentation (S3 / S4)

Plan: `docs/OPEN_DIRECTIONS.md` S3 and S4. VGG fc7 + ECAPA-BTC 192 are our best bridge pair; both are the LAST layers of
their networks, pushed toward recognition. The earlier layers may keep more of the soft attributes a voice predicts.
Both encoders are the exact ones the organisers used (`flag_extract.VGGFace`, `flag_extract.OrganiserVoice`, cos 1.0000
with their csv), so every array here lives beside the features the bridge already uses.

| stream | layer | dim | direction |
|---|---|---:|---|
| `face_btcface` | fc7 post-ReLU = the organiser feature (checked against their csv) | 4096 | sanity |
| `face_btcfacefc6` | fc6 post-ReLU | 4096 | S3 |
| `face_btcfacepool5` | pool5 averaged over its 7 x 7 grid | 512 | S3 |
| `face_btcfaceflip` | fc7 of the horizontally mirrored image | 4096 | S4 (average with fc7) |
| `voice_btcvoice` | 192 output = the organiser feature (checked against their csv) | 192 | sanity |
| `voice_btcvoiceasp` | attentive-statistics pooling, the layer before the 192 projection | 3072 | S3 |
| `voice_btcvoicetta` | mean of the 192 output over the whole file + two 80 % crops | 192 | S4 |

**Settings:** Accelerator **GPU T4 x2** (one GPU used), **Internet ON** (VGG-Face 580 MB from robots.ox.ac.uk, the
organiser's ECAPA from Hugging Face; cached in `/tmp`, not in the output). fp32 throughout.

**Inputs:** exactly as FLAG_10: the raw v4 dataset; optional MAV-Celeb v1 / v2 + `flag2027-ext-meta` for Urdu / Hindi rows.
The raw v4 dataset also carries the organiser's `train_English_{faces,voices}.csv`, used for the reproduction check.

**Resume after a failure:** attach this notebook's earlier output (`flag2027-feats-layers`); finished encoders are skipped.

**Outputs:** `/kaggle/working/feats_layers/` -> dataset **`flag2027-feats-layers`** (~1 GB: v4 fp32, v1/v2 fp16).
Expected time on a T4: downloads ~3 min, VGG-Face ~10 min, ECAPA with its TTA crops ~30 min.

In [ ]:
import os, sys
from pathlib import Path
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_models_cache"))   # checkpoints stay out of the output
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| working", WORKDIR, "| cache", CACHE)

In [ ]:
# Writes the helper modules next to this notebook (no .py upload needed).
import base64
open('flag_lib.py', 'wb').write(base64.b64decode(''.join('''
...
cmV0dXJuIHBhdGgK
'''.split())))
open('flag_extract.py', 'wb').write(base64.b64decode(''.join('''
...
IHJvb3QKICAgIHJldHVybiBOb25lCgo=
'''.split())))
open('agegender.py', 'wb').write(base64.b64decode(''.join('''
IiIiQWdlLWdlbmRlciBmZWF0dXJlIGV4dHJhY3Rpb24g4oCUIHRoZSBhdXhpbGlhcnkgc3RyZWFtcyBvZiB0aGUgRkFNRSAyMDI2
IHdpbm5pbmcgc3lzdGVtLgoKVGhlaXIgc3lzdGVtIGNvbmNhdGVuYXRlcywgcGVyIG1vZGFsaXR5LCBhIGdlbmVyYWwgaWRlbnRp
dHkgZW1iZWRkaW5nIHdpdGggYW4gYWdlLWdlbmRlciBlbWJlZGRpbmcKYmVmb3JlIHRoZSBzaW5nbGUgbGluZWFyIG1hcHBpbmcg
bGF5ZXI6CgogICAgZmFjZSAgOiBWR0dGYWNlIDQwOTYtZCAgICsrICBWaVQgYWdlLWdlbmRlciA3NjgtZCAgICAtPiBsaW5lYXIg
LT4gMTkyCiAgICB2b2ljZSA6IEVDQVBBICA2MTQ0LWQgICAgKysgIEVDQVBBIGFnZS1nZW5kZXIgMTUzNi1kIC0+IGxpbmVhciAt
PiAxOTIKClRoZWlyIGFnZS1nZW5kZXIgRUNBUEEgKDYuNU0gcGFyYW1zLCAxNTM2LWQpIGlzIG5vdCBwdWJsaWMsIHNvIHdlIHVz
ZSB0aGUgY2xvc2VzdCBwdWJsaWMgc3RhbmQtaW5zOgoKICAgIHZvaWNlIDogYXVkZWVyaW5nL3dhdjJ2ZWMyLWxhcmdlLXJvYnVz
dC0yNC1mdC1hZ2UtZ2VuZGVyICAtPiAxMDI0LWQgcG9vbGVkIGhpZGRlbiBzdGF0ZQogICAgICAgICAgICAoZmluZS10dW5lZCBv
biBhR2VuZGVyLCBDb21tb24gVm9pY2UsIFRJTUlULCBWb3hDZWxlYjIpCiAgICAgICAgICAgIExJQ0VOQ0U6IENDIEJZLU5DLVNB
IDQuMCwgbm9uLWNvbW1lcmNpYWwuIEZpbmUgZm9yIGEgcmVzZWFyY2ggY2hhbGxlbmdlLCBidXQgaXQgTVVTVCBiZQogICAgICAg
ICAgICBkZWNsYXJlZCBpbiB0aGUgc3lzdGVtIGRlc2NyaXB0aW9uLgogICAgZmFjZSAgOiBuYXRlcmF3L3ZpdC1hZ2UtY2xhc3Np
ZmllciAtPiA3NjgtZCBDTFMgZW1iZWRkaW5nIChWaVQtYmFzZSwgZXhhY3RseSB0aGVpciBkaW1lbnNpb24pCgpPdXIgb3duIEVE
QSBhbHJlYWR5IHNob3dlZCB0aGVzZSBhdHRyaWJ1dGVzIGFyZSBzdHJvbmdseSBwcmVzZW50IGFuZCBjdXJyZW50bHkgdXNlZCBp
bXBsaWNpdGx5OgphIGxpbmVhciBnZW5kZXIgcHJvYmUgcmVhY2hlcyA5MyUgb24gZmFjZSBhbmQgOTIlIG9uIHZvaWNlIChFREEt
NCkuIE1ha2luZyB0aGVtIGFuIGV4cGxpY2l0LApzZXBhcmF0ZSBzdHJlYW0gaXMgd2hhdCBsZXRzIHRoZSBpZGVudGl0eSBwYXJ0
IG9mIHRoZSBlbWJlZGRpbmcgc3RvcCBjYXJyeWluZyB0aGVtLgoKVXNlZCBieSB0aGUgS2FnZ2xlIGV4dHJhY3Rpb24gbm90ZWJv
b2s7IGtlcHQgaW4gaXRzIG93biBmaWxlIHNvIHRoZSBub3RlYm9vayBzdGF5cyByZWFkYWJsZS4KIiIiCmZyb20gX19mdXR1cmVf
XyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5u
CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0g
dm9pY2UKY2xhc3MgX0hlYWQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWcsIG51bV9sYWJlbHMpOgog
ICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZGVuc2UgPSBubi5MaW5lYXIoY29uZmlnLmhpZGRlbl9zaXpl
LCBjb25maWcuaGlkZGVuX3NpemUpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChjb25maWcuZmluYWxfZHJvcG91
dCkKICAgICAgICBzZWxmLm91dF9wcm9qID0gbm4uTGluZWFyKGNvbmZpZy5oaWRkZW5fc2l6ZSwgbnVtX2xhYmVscykKCiAgICBk
ZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4ID0gc2VsZi5kcm9wb3V0KHgpCiAgICAgICAgeCA9IHRvcmNoLnRhbmgoc2Vs
Zi5kZW5zZSh4KSkKICAgICAgICByZXR1cm4gc2VsZi5vdXRfcHJvaihzZWxmLmRyb3BvdXQoeCkpCgoKZGVmIGJ1aWxkX3ZvaWNl
X2FnZV9nZW5kZXIoZGV2aWNlKToKICAgICIiIlJldHVybnMgKG1vZGVsLCBwcm9jZXNzb3IpLiBtb2RlbCh4KSAtPiAocG9vbGVk
XzEwMjQsIGFnZV9sb2dpdCwgZ2VuZGVyX3Byb2JzKS4iIiIKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBXYXYyVmVjMlBy
b2Nlc3NvcgogICAgZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLndhdjJ2ZWMyLm1vZGVsaW5nX3dhdjJ2ZWMyIGltcG9ydCBXYXYy
VmVjMk1vZGVsLCBXYXYyVmVjMlByZVRyYWluZWRNb2RlbAoKICAgIGNsYXNzIEFnZUdlbmRlck1vZGVsKFdhdjJWZWMyUHJlVHJh
aW5lZE1vZGVsKToKICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgY29uZmlnKToKICAgICAgICAgICAgc3VwZXIoKS5fX2luaXRf
Xyhjb25maWcpCiAgICAgICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnCiAgICAgICAgICAgIHNlbGYud2F2MnZlYzIgPSBXYXYy
VmVjMk1vZGVsKGNvbmZpZykKICAgICAgICAgICAgc2VsZi5hZ2UgPSBfSGVhZChjb25maWcsIDEpCiAgICAgICAgICAgIHNlbGYu
Z2VuZGVyID0gX0hlYWQoY29uZmlnLCAzKQogICAgICAgICAgICBzZWxmLnBvc3RfaW5pdCgpICAgICAgICAjIG5vdCBpbml0X3dl
aWdodHMoKTogdHJhbnNmb3JtZXJzID49IDUgc2V0cyBhbGxfdGllZF93ZWlnaHRzX2tleXMgaGVyZQoKICAgICAgICBkZWYgZm9y
d2FyZChzZWxmLCBpbnB1dF92YWx1ZXMpOgogICAgICAgICAgICBoID0gc2VsZi53YXYydmVjMihpbnB1dF92YWx1ZXMpWzBdLm1l
YW4oZGltPTEpICAgICAgICMgbWVhbi1wb29sIG92ZXIgdGltZSAtPiAxMDI0LWQKICAgICAgICAgICAgcmV0dXJuIGgsIHNlbGYu
YWdlKGgpLCB0b3JjaC5zb2Z0bWF4KHNlbGYuZ2VuZGVyKGgpLCBkaW09MSkKCiAgICBuYW1lID0gImF1ZGVlcmluZy93YXYydmVj
Mi1sYXJnZS1yb2J1c3QtMjQtZnQtYWdlLWdlbmRlciIKICAgIHByb2MgPSBXYXYyVmVjMlByb2Nlc3Nvci5mcm9tX3ByZXRyYWlu
ZWQobmFtZSkKICAgIG1vZGVsID0gQWdlR2VuZGVyTW9kZWwuZnJvbV9wcmV0cmFpbmVkKG5hbWUpLnRvKGRldmljZSkuZXZhbCgp
CiAgICByZXR1cm4gbW9kZWwsIHByb2MKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiB2b2ljZV9hZ2VfZ2VuZGVyX2VtYmVkKG1vZGVs
LCBwcm9jLCBwYXRocywgcm9vdCwgZGV2aWNlLCBicz04LCBtYXhfc2VjPTguMCk6CiAgICAiIiJNZWFuLXBvb2xlZCAxMDI0LWQg
aGlkZGVuIHN0YXRlIHBsdXMgdGhlIGFnZS9nZW5kZXIgcHJlZGljdGlvbnMsIGluIGBwYXRoc2Agb3JkZXIuIiIiCiAgICBpbXBv
cnQgc291bmRmaWxlIGFzIHNmCiAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCiAgICBlbWIsIGFnZSwgZ2VuID0gW10sIFtd
LCBbXQogICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVuKHBhdGhzKSwgYnMpOgogICAgICAgIHNpZ3MgPSBbXQogICAgICAgIGZvciBw
IGluIHBhdGhzW3M6cyArIGJzXToKICAgICAgICAgICAgdywgc3IgPSBzZi5yZWFkKHN0cihQYXRoKHJvb3QpIC8gcCksIGR0eXBl
PSJmbG9hdDMyIiwgYWx3YXlzXzJkPVRydWUpCiAgICAgICAgICAgIGFzc2VydCBzciA9PSAxNjAwMCwgc3IKICAgICAgICAgICAg
c2lncy5hcHBlbmQod1s6IGludChtYXhfc2VjICogc3IpLCAwXSkgICAgICAgICAgICAgICAjIGNhcCBsZW5ndGg6IHRoaXMgbW9k
ZWwgaXMgaGVhdnkKICAgICAgICBMID0gbWF4KGxlbih3KSBmb3IgdyBpbiBzaWdzKQogICAgICAgIHggPSBucC56ZXJvcygobGVu
KHNpZ3MpLCBMKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBmb3IgaSwgdyBpbiBlbnVtZXJhdGUoc2lncyk6CiAgICAgICAg
ICAgIHhbaSwgOiBsZW4odyldID0gdwogICAgICAgIHggPSBwcm9jKGxpc3QoeCksIHNhbXBsaW5nX3JhdGU9MTYwMDAsIHJldHVy
bl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSkuaW5wdXRfdmFsdWVzCiAgICAgICAgaCwgYSwgZyA9IG1vZGVsKHgudG8oZGV2
aWNlKSkKICAgICAgICBlbWIuYXBwZW5kKGguY3B1KCkubnVtcHkoKSk7IGFnZS5hcHBlbmQoYS5jcHUoKS5udW1weSgpKTsgZ2Vu
LmFwcGVuZChnLmNwdSgpLm51bXB5KCkpCiAgICAgICAgaWYgcyAlIChicyAqIDI1KSA9PSAwOgogICAgICAgICAgICBwcmludChm
IiAgIHZvaWNlLWFnIHtzfS97bGVuKHBhdGhzKX0iLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKGVtYiks
IG5wLmNvbmNhdGVuYXRlKGFnZSksIG5wLmNvbmNhdGVuYXRlKGdlbikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t
LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBmYWNlCmRlZiBidWlsZF9mYWNlX2FnZShkZXZpY2UsIG5h
bWU9Im5hdGVyYXcvdml0LWFnZS1jbGFzc2lmaWVyIik6CiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b0ltYWdlUHJv
Y2Vzc29yLCBBdXRvTW9kZWxGb3JJbWFnZUNsYXNzaWZpY2F0aW9uCiAgICBwcm9jID0gQXV0b0ltYWdlUHJvY2Vzc29yLmZyb21f
cHJldHJhaW5lZChuYW1lKQogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JJbWFnZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5l
ZChuYW1lKS50byhkZXZpY2UpLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsLCBwcm9jCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgZmFj
ZV9hZ2VfZW1iZWQobW9kZWwsIHByb2MsIHBhdGhzLCByb290LCBkZXZpY2UsIGJzPTMyKToKICAgICIiIjc2OC1kIENMUyBlbWJl
ZGRpbmcgZnJvbSB0aGUgVmlUIGFnZSBjbGFzc2lmaWVyLCBpbiBgcGF0aHNgIG9yZGVyLiIiIgogICAgZnJvbSBQSUwgaW1wb3J0
IEltYWdlCiAgICBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCiAgICBvdXQgPSBbXQogICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVu
KHBhdGhzKSwgYnMpOgogICAgICAgIGltZ3MgPSBbSW1hZ2Uub3BlbihQYXRoKHJvb3QpIC8gcCkuY29udmVydCgiUkdCIikgZm9y
IHAgaW4gcGF0aHNbczpzICsgYnNdXQogICAgICAgIHggPSBwcm9jKGltYWdlcz1pbWdzLCByZXR1cm5fdGVuc29ycz0icHQiKS5w
aXhlbF92YWx1ZXMudG8oZGV2aWNlKQogICAgICAgIGggPSBtb2RlbC5iYXNlX21vZGVsKHgpLmxhc3RfaGlkZGVuX3N0YXRlWzos
IDBdICAgICAgICAgICAjIENMUyB0b2tlbgogICAgICAgIG91dC5hcHBlbmQoaC5jcHUoKS5udW1weSgpKQogICAgICAgIGlmIHMg
JSAoYnMgKiAyNSkgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiIgICBmYWNlLWFnIHtzfS97bGVuKHBhdGhzKX0iLCBmbHVzaD1U
cnVlKQogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKG91dCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t
LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjaGVjawpkZWYgZ2VuZGVyX2FncmVlbWVudChnZW5fcHJvYnMsIGdl
bmRlcl9sYWJlbHMpOgogICAgIiIiU2FuaXR5IGNoZWNrIGFnYWluc3Qgb3VyIG93biBsYWJlbHM6IHRoZSBtb2RlbCdzIG1hbGUv
ZmVtYWxlIGNhbGwgc2hvdWxkIG1vc3RseSBhZ3JlZS4KCiAgICBnZW5fcHJvYnMgY29sdW1ucyBhcmUgKGZlbWFsZSwgbWFsZSwg
Y2hpbGQpOiBjb25maWcuanNvbiBpZDJsYWJlbC4gVGhlIG1vZGVsIGNhcmQncyBwcm9zZSBzYXlzCiAgICAiY2hpbGQsIGZlbWFs
ZSwgb3IgbWFsZSIsIGJ1dCB0aGUgY29uZmlnIGFuZCBvdXIgdjQgY2hlY2sgKGYgLT4gY29sdW1uIDAsIG0gLT4gY29sdW1uIDEp
IGFncmVlLgogICAgIiIiCiAgICBwcmVkID0gbnAud2hlcmUoZ2VuX3Byb2JzWzosIDFdID49IGdlbl9wcm9ic1s6LCAwXSwgIm0i
LCAiZiIpCiAgICByZXR1cm4gZmxvYXQoKHByZWQgPT0gbnAuYXNhcnJheShnZW5kZXJfbGFiZWxzKSkubWVhbigpKQo=
'''.split())))
open('flag_models.py', 'wb').write(base64.b64decode(''.join('''
...
dHkK
'''.split())))
print('written:', ['flag_lib.py', 'flag_extract.py', 'agegender.py', 'flag_models.py'])

## 1. Setup, v4 index (row order = organiser txt), v1/v2 rows

In [ ]:
# -U matters: Kaggle preinstalls an old speechbrain (< 1.1.1 breaks on torchaudio 2.9+), as FLAG_04 / FLAG_08.
if not LOCAL:
    !pip -q install -U "speechbrain>=1.1.1" soundfile 2>&1 | tail -2
import numpy as np, pandas as pd, torch, json, time, speechbrain
print("torch", torch.__version__, "| speechbrain", speechbrain.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import flag_extract as X, flag_models as M
OUT = WORKDIR / "feats_layers"; OUT.mkdir(exist_ok=True)
FACE, VOICE = ["btcface"], ["btcvoice"]  # the organiser's VGG-Face / ECAPA, with their other layers and TTA
ORDER = FACE + VOICE
# ONE walk over /kaggle/input that skips faces/ and voices/ folders: an extracted v1/v2 holds hundreds of thousands
# of files, and every rglob over it costs minutes. Everything below looks things up in SCAN instead.
SCAN = M.scan_inputs(IN)
S = M.v4_index(SCAN, WORKDIR / "raw")
meta = SCAN["files"].get("meta_file_train_set.csv")
assert meta, "meta_file_train_set.csv (train gender labels) not found in the inputs: attach the raw v4 dataset"
X.copy_metadata(meta[0].parent, S, OUT)

# MAV-Celeb v1/v2 validation rows (optional): the archive/folder + the existing meta csv that fixes the row order.
# Several copies (capped feats_ext vs feats_ext_full) -> take the largest: it is the superset, joined on `path` later.
EXT = {}
for name in ["v1_complete", "v2_complete"]:
    metas = sorted(SCAN["files"].get(f"ext_{name}_voice_meta.csv", []), key=lambda p: -p.stat().st_size)
    src = M.ext_source(SCAN, name)
    if metas and src is not None:
        try:
            print(f"{name}: rows from {metas[0].parent}")
            EXT[name] = M.ExtRows(src, name, metas[0].parent)
        except Exception as e:                    # optional source: a bad zip / missing row must not stop the run
            print(f"{name}: skipped, cannot read its rows from {src} ({type(e).__name__}: {e})")
    else:
        print(f"{name}: skipped (meta csv found: {bool(metas)}, source found: {src is not None})")
print("v4 rows per split:", {k: len(v["wav"]) for k, v in S.items()}, "| external:", list(EXT))

# Resume: attach an earlier version's output (flag2027-feats-layers) and its arrays are copied here, so only the
# encoders still missing run (e.g. after an encoder failed). Nothing to copy on a first run.
SEEDED = M.seed_from_previous(SCAN, OUT, ORDER)

## 2. Extract: one encoder at a time, v4 then v1/v2

In [ ]:
CHECKS, FAILED = {}, {}
face_probe = [S["train"]["root"] / p for p in S["train"]["jpg"][:6]]
voice_probe = [X.load_audio(S["no_gender/Bangla"]["root"] / p) for p in S["no_gender/Bangla"]["wav"][:4]]
for name in ORDER:
    kind = "face" if name in FACE else "voice"
    t0 = time.time()
    if M.done(name, kind, S, EXT, OUT):
        print(f"== {name}: every array already exists -> skipped (not even loaded)")
        continue
    try:
        enc = M.build(name, CACHE)
        CHECKS[name] = M.check_face(enc, face_probe) if kind == "face" else M.check_voice(enc, voice_probe)
        print(name, "checks:", CHECKS[name])
        assert min(CHECKS[name].values()) > 0.995, f"{name}: batching / fp16 changes the embedding {CHECKS[name]}"
        M.extract_v4(name, enc, S, OUT, kind)
        for src_name, rows in EXT.items():
            M.extract_ext(name, enc, rows, src_name, OUT, kind)
    except Exception as e:                        # an optional encoder must not kill the run
        FAILED[name] = f"{type(e).__name__}: {e}"
        print(f"!! {name} FAILED -> skipped: {FAILED[name]}")
    enc = None
    if name != "ibv":                              # keep ImageBind loaded for the audio pass right after
        M.release()
    print(f"== {name} done in {time.time() - t0:.0f}s")
print("failed:", FAILED or "none")

## 3. Checks: reproduction of the organiser csv, identity EER per layer

In [ ]:
ORG = M.check_organiser(OUT, SCAN)
print("re-extracted fc7 / 192 vs the organiser csv (must be ~1.0000):", ORG)
tr = pd.read_csv(OUT / "index_train.txt", sep=" ", header=None)
spk = tr[4].values
gmap = dict(pd.read_csv(OUT / "meta_file_train_set.csv", header=None).values)
P, SANITY = M.probe(OUT, spk, gmap, seeds=(1,) if LOCAL else (1, 2, 3), n=200 if LOCAL else 3000)
SANITY["organiser_csv_cos"] = ORG
P.to_csv(OUT / "probe.csv", index=False)
print(P.to_string(index=False))
# Identity EER per layer (same modality, train): earlier layers should be worse at identity than fc7 / 192.
# That is expected and is not the test; the bridge harness (Experiment/MODEL-01) decides whether they help face <-> voice.

## 4. Manifest + clean-up

In [ ]:
import shutil
man = X.write_manifest(OUT, dict(checks=CHECKS, failed=FAILED, sanity=SANITY, external=list(EXT),
                                 encoders=ORDER, audio=X.AUDIO_STATS))
(OUT / "probe.json").write_text(json.dumps(dict(sanity=SANITY, checks=CHECKS, failed=FAILED), indent=1))
for p in [WORKDIR / "raw"]:
    shutil.rmtree(p, ignore_errors=True)
tot = sum(p.stat().st_size for p in OUT.glob("*")) / 1e9
print(f"\n{len(man['files'])} arrays, {tot:.2f} GB in {OUT}")
print("NEXT: Save Version (Save & Run All) -> open the version -> Output -> 'New Dataset' -> name it flag2027-feats-layers")